In [14]:
import json
import logging
import os
import time
from datetime import datetime, timedelta

import pandas as pd
import pyotp
import requests
from SmartApi import SmartConnect

In [ ]:
# CONFIG — fill these in
# Groww API credentials.
# CONFIG — fill these in M50848322
API_KEY = "3LjGsQyt"
CLIENT_ID = "M50848322"
PASSWORD = "8581" 
TOTP_SECRET = "C4P6OKR4CY3QHB6DPTYGWLUIC4"     # Base32 secret from SmartAPI TOTP setup

TELEGRAM_BOT_TOKEN = "8805272234:AAFVqOaf2mrYzjqCb7zufjkaWGdwR39f460"
TELEGRAM_CHAT_ID = "926442490"

DRY_RUN = False       # True = simulate orders only (no real order placed). Set False to go live.
PRODUCT_TYPE = "INTRADAY"   # INTRADAY / DELIVERY / CARRYFORWARD (Angel One naming)
ORDER_VARIETY = "NORMAL"
LOOP_SLEEP_SECONDS = 30      # how often the main loop ticks
STATE_FILE = "bot_state.json"

MARKET_OPEN = "09:15"
MARKET_CLOSE = "23:30"


# 10-Minute candles are fetched every 10 minutes EXCEPT at the ":15" mark, because ":15" is
# already handled by the 1-Hour fetch above and would otherwise just re-read the same
# just-closed 1H candle boundary (e.g. 9:25, 9:35, 9:45, 9:55, 10:05, 10:25, 10:35 ... — never 9:15/10:15/...).
TEN_MIN_FETCH_MINUTES = {5,15,25,35,45,55}

In [16]:
# LOGGING

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("ha_bot.log"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger("ha_bot")

In [ ]:
WATCHLIST = [
    {
        "symbol": "SENSEX",
        "exchange": "BSE",
        "token":"99919000",
    },
      {
        "symbol": "ELECTWKCOM",
        "exchange": "MCX",
        "token": "446",
    }
    # Add more symbols here...
]

In [18]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("ha_bot.log"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger("ha_bot")

In [19]:
# TELEGRAM
def send_telegram(message: str):
    try:
        url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
        requests.post(url, data={"chat_id": TELEGRAM_CHAT_ID, "text": message}, timeout=10)
    except Exception as e:
        log.error(f"Telegram send failed: {e}")

In [20]:
# STATE PERSISTENCE
def default_symbol_state():
    return {
        "position": None,                     # dict with entry_price, quantity, stoploss_price, entry_time
        "pending_signal": None,                # "BUY" or "SELL" (1H bias confirmed, waiting for 10min trigger)
        "pending_signal_1h_close_time": None,  # ISO timestamp of the 1H candle close that set the bias
        "last_processed_1h_time": None,        # avoid re-evaluating the same 1H candle repeatedly
        "last_processed_10m_time": None,       # avoid re-evaluating the same 10min candle repeatedly
    }


def reset_symbol_state_keep_position(sym_state):
    """Reset every tracked variable EXCEPT 'position'. Called for every symbol right before
    each fixed-time 1H candle fetch (9:15, 10:15, 11:15, 12:15, 1:15, 2:15)."""
    sym_state["position"]=None
    sym_state["pending_signal"] = None
    sym_state["pending_signal_1h_close_time"] = None
    sym_state["last_processed_1h_time"] = None
    sym_state["last_processed_10m_time"] = None


def load_state():
    if os.path.exists(STATE_FILE):
        try:
            with open(STATE_FILE, "r") as f:
                data = json.load(f)
            log.info("Loaded existing state from disk (resuming after restart).")
            return data
        except Exception as e:
            log.error(f"Failed to load state file, starting fresh: {e}")
    return {item["symbol"]: default_symbol_state() for item in WATCHLIST}


def save_state(state):
    try:
        with open(STATE_FILE, "w") as f:
            json.dump(state, f, indent=2, default=str)
    except Exception as e:
        log.error(f"Failed to save state: {e}")

In [21]:
# ANGEL ONE LOGIN
def login():
    obj = SmartConnect(api_key=API_KEY)
    totp = pyotp.TOTP(TOTP_SECRET).now()
    data = obj.generateSession(CLIENT_ID, PASSWORD, totp)
    if not data.get("status"):
        raise RuntimeError(f"Login failed: {data}")
    log.info("Logged in to Angel One SmartAPI.")
    return obj

In [22]:
def fetch_candles(smart_api, token, exchange, interval, lookback_minutes):
    """
    interval: "ONE_HOUR" or "TEN_MINUTE" (Angel One interval codes)
    Returns a DataFrame with columns: time, open, high, low, close, volume
    """
    to_date = datetime.now()
    from_date = to_date - timedelta(minutes=lookback_minutes)
    params = {
        "exchange": exchange,
        "symboltoken": token,
        "interval": interval,
        "fromdate": from_date.strftime("%Y-%m-%d %H:%M"),
        "todate": to_date.strftime("%Y-%m-%d %H:%M"),
    }
    
    for attempt in range(3):
        try:
            resp = smart_api.getCandleData(params)
            if resp.get("status") and resp.get("data"):
                df = pd.DataFrame(
                    resp["data"], columns=["time", "open", "high", "low", "close", "volume"]
                )
                
                df["time"] = pd.to_datetime(df["time"])
                for col in ["open", "high", "low", "close", "volume"]:
                    df[col] = df[col].astype(float)
                print(df)
                return df
                
            else:
                log.warning(f"Candle fetch returned no data (attempt {attempt+1}): {resp}")
        except Exception as e:
            log.error(f"Candle fetch error (attempt {attempt+1}): {e}")
        time.sleep(1)
    return None


def to_heikin_ashi(df):
    """Convert a normal OHLC dataframe to Heikin-Ashi OHLC."""
    ha = df.copy().reset_index(drop=True)
    ha["ha_close"] = (df["open"] + df["high"] + df["low"] + df["close"]) / 4.0
    ha_open = [(df["open"].iloc[0] + df["close"].iloc[0]) / 2.0]
    for i in range(1, len(df)):
        ha_open.append((ha_open[i - 1] + ha["ha_close"].iloc[i - 1]) / 2.0)
    ha["ha_open"] = ha_open
    ha["ha_high"] = ha[["ha_open", "ha_close"]].join(df["high"]).max(axis=1)
    ha["ha_low"] = ha[["ha_open", "ha_close"]].join(df["low"]).min(axis=1)
    return ha


def candle_color(open_price, close_price):
    return "GREEN" if close_price >= open_price else "RED"


def is_last_candle_completed(df, interval_minutes):
    """
    The LAST row returned may still be the currently-forming (incomplete) candle. A candle is
    only "completed" once its close-time (open-time + interval) has passed.
    Returns True if df.iloc[-2] is a fully completed candle we can safely evaluate,
    and there are at least 2 rows.
    """
    if df is None or len(df) < 2:
        return False
    last_candle_open = df["time"].iloc[-1]
    last_candle_close_time = last_candle_open + timedelta(minutes=interval_minutes)
    now = datetime.now(last_candle_open.tzinfo) if last_candle_open.tzinfo else datetime.now()
    return now >= last_candle_close_time


def get_last_completed(df):
    """Return the second-to-last row = latest fully closed candle (last row may be forming)."""
    return df.iloc[-2]

In [23]:
"""import pandas as pd
def last_six_candle():
  # df=fetch_candles(smart_api, symbol_info["token"], symbol_info["exchange"], "ONE_MINUTE", 10 * 10)
,
    {
        "symbol": "RELIANCE-EQ",
        "exchange": "NSE",
        "token":"2885",
    },
    {
        "symbol": "ICICIBANK-EQ",
        "exchange": "NSE",
        "token":"4963 ",
    },{
        "symbol": "TCS-EQ",
        "exchange": "NSE",
        "token":"11536 ",
    }
  candles = [
    ["2026-07-19 09:15:00", 100.0, 101.0, 95.0, 96.0, 10000],
    ["2026-07-19 09:25:00", 96.0, 97.0, 91.0, 92.0, 11000],
    ["2026-07-19 09:35:00", 92.0, 93.0, 87.0, 88.0, 12000],
    ["2026-07-19 09:45:00", 88.0, 89.0, 83.0, 84.0, 13000],
    ["2026-07-19 09:55:00", 84.0, 85.0, 79.0, 80.0, 14000],
    ["2026-07-19 10:05:00", 80.0, 81.0, 75.0, 76.0, 15000],
    ["2026-07-19 10:15:00", 76.0, 77.0, 71.0, 72.0, 16000],
    ["2026-07-19 10:25:00", 72.0, 73.0, 67.0, 68.0, 17000],

    # Second-last candle (becomes green in Heikin-Ashi)
    ["2026-07-19 10:35:00", 68.0, 82.0, 67.0, 80.0, 18000],

    # Last candle
    ["2026-07-19 10:45:00", 80.0, 81.0, 78.0, 79.0, 19000],
]
  df = pd.DataFrame(candles,columns=["time", "open", "high", "low", "close", "volume"]
  )
  
  df["time"] = pd.to_datetime(df["time"])
  
  heiken_ashi=to_heikin_ashi(df)
  c1=heiken_ashi.iloc[-3]
  c2=heiken_ashi.iloc[-4]
  c3=heiken_ashi.iloc[-5]
  c4=heiken_ashi.iloc[-6]
  c5=heiken_ashi.iloc[-7]
  c6=heiken_ashi.iloc[-8]
  cl_color=candle_color(c1["ha_open"],c1["ha_close"])
  c2_color=candle_color(c2["ha_open"],c2["ha_close"])
  c3_color=candle_color(c3["ha_open"],c3["ha_close"])
  c4_color=candle_color(c4["ha_open"],c4["ha_close"])
  c5_color=candle_color(c5["ha_open"],c5["ha_close"])
  c6_color=candle_color(c6["ha_open"],c6["ha_close"])
  print(cl_color,c2_color,c3_color,c4_color,c5_color,c6_color)
  c0=heiken_ashi.iloc[-2]
  c0_color=candle_color(c0["ha_open"],c0["ha_close"])
  c0_n_color=candle_color(c0["open"],c0["close"])
  if(c0_color=="RED" and c0_n_color=="RED" ):
    

    if(cl_color == "GREEN" and c2_color == "GREEN" and c3_color == "GREEN" and c4_color == "GREEN" and c5_color == "GREEN" and c6_color == "GREEN"):
      call="SELL"
      return call
      
  if(c0_color=="GREEN" and c0_n_color=="GREEN" ):
    if(cl_color == "RED" and c2_color == "RED" and c3_color == "RED" and c4_color == "RED" and c5_color == "RED" and c6_color == "RED"):
      call="BUY"
      return call
    
      
  
p1=last_six_candle()

"""

'import pandas as pd\ndef last_six_candle():\n  # df=fetch_candles(smart_api, symbol_info["token"], symbol_info["exchange"], "ONE_MINUTE", 10 * 10)\n,\n    {\n        "symbol": "RELIANCE-EQ",\n        "exchange": "NSE",\n        "token":"2885",\n    },\n    {\n        "symbol": "ICICIBANK-EQ",\n        "exchange": "NSE",\n        "token":"4963 ",\n    },{\n        "symbol": "TCS-EQ",\n        "exchange": "NSE",\n        "token":"11536 ",\n    }\n  candles = [\n    ["2026-07-19 09:15:00", 100.0, 101.0, 95.0, 96.0, 10000],\n    ["2026-07-19 09:25:00", 96.0, 97.0, 91.0, 92.0, 11000],\n    ["2026-07-19 09:35:00", 92.0, 93.0, 87.0, 88.0, 12000],\n    ["2026-07-19 09:45:00", 88.0, 89.0, 83.0, 84.0, 13000],\n    ["2026-07-19 09:55:00", 84.0, 85.0, 79.0, 80.0, 14000],\n    ["2026-07-19 10:05:00", 80.0, 81.0, 75.0, 76.0, 15000],\n    ["2026-07-19 10:15:00", 76.0, 77.0, 71.0, 72.0, 16000],\n    ["2026-07-19 10:25:00", 72.0, 73.0, 67.0, 68.0, 17000],\n\n    # Second-last candle (becomes green in 

In [24]:
#MARKET HOUR

def market_is_open():
    now = datetime.now()
    open_t = datetime.strptime(MARKET_OPEN, "%H:%M").replace(
        year=now.year, month=now.month, day=now.day
    )
    close_t = datetime.strptime(MARKET_CLOSE, "%H:%M").replace(
        year=now.year, month=now.month, day=now.day
    )
    return open_t <= now <= close_t and now.weekday() < 5

In [ ]:
# CORE STRATEGY LOGIC — per symbol, per tick
def process_symbol(smart_api, symbol_info, state):
    symbol = symbol_info["symbol"]
    sym_state = state[symbol]
    df_10m = fetch_candles(smart_api, symbol_info["token"],symbol_info["exchange"], "TEN_MINUTE", 10 * 8)
    """if not is_last_candle_completed(df_10m, 10):
      print("HGH")
      return"""
    if df_10m is None:
        log.warning(f"{symbol}: No candle data.")
        print("HGH")
        return

    if len(df_10m) < 2:
        log.warning(f"{symbol}: Not enough candles.")
        print("HGH")
        return

    last_10m = get_last_completed(df_10m)
    last_10m_time = str(last_10m["time"])
    #if sym_state["last_processed_10m_time"] == last_10m_time:
        #return  # already evaluated this 10-min candle

    sym_state["last_processed_10m_time"] = last_10m_time
    heiken_ashi=to_heikin_ashi(df_10m)
    c1=heiken_ashi.iloc[-3]
    c2=heiken_ashi.iloc[-4]
    c3=heiken_ashi.iloc[-5]
    c4=heiken_ashi.iloc[-6]
    c5=heiken_ashi.iloc[-7]
    c6=heiken_ashi.iloc[-8]
    cl_color=candle_color(c1["ha_open"],c1["ha_close"])
    c2_color=candle_color(c2["ha_open"],c2["ha_close"])
    c3_color=candle_color(c3["ha_open"],c3["ha_close"])
    c4_color=candle_color(c4["ha_open"],c4["ha_close"])
    c5_color=candle_color(c5["ha_open"],c5["ha_close"])
    c6_color=candle_color(c6["ha_open"],c6["ha_close"])
    
    c0=heiken_ashi.iloc[-2]
    c0_color=candle_color(c0["ha_open"],c0["ha_close"])
    c0_n_color=candle_color(c0["open"],c0["close"])
    #print(c0_color,c0_n_color,cl_color,c2_color,c3_color,c4_color,c5_color,c6_color)

   
    if(c0_color=="RED" and c0_n_color=="RED" ):

        if(cl_color == "GREEN" and c2_color == "GREEN" and c3_color == "GREEN" and c4_color == "GREEN" and c5_color == "GREEN" and c6_color == "GREEN"):
          #print(f"Sell Signal for {symbol}.Current candle is Red and Previous Six candle is Green")
          send_telegram(f"Sell Signal for {symbol}.Current candle is Red and Previous Six candle is Green")
      
    if(c0_color=="GREEN" and c0_n_color=="GREEN" ):
        if(cl_color == "RED" and c2_color == "RED" and c3_color == "RED" and c4_color == "RED" and c5_color == "RED" and c6_color == "RED"):
          #print(f"Buy Signal for {symbol}.Current candle is Green and Previous Six candle is Red")
          send_telegram(f"Buy Signal for {symbol}.Current candle is Green and Previous Six candle is Red")





In [ ]:
def main():
    send_telegram("🤖 Algo trading bot started (Angel One SmartAPI). Watching: "
           + ", ".join(c["symbol"] for c in WATCHLIST
           ))
    log.info(f"Starting bot. DRY_RUN={DRY_RUN}")
    if DRY_RUN:
        log.warning("Running in DRY_RUN mode — no real orders will be placed. Set DRY_RUN=False to go live.")

    smart_api = login()
    state = load_state()
    last_10m_marker = None

    while True:
        try:
            if not market_is_open():
                log.info("Market closed. Sleeping 5 minutes.")
                send_telegram("Market is closed")
                time.sleep(300)
                continue
            #wait_for_next_candle()
            now = datetime.now()
            current_hm = now.strftime("%H:%M")
            if now.minute in TEN_MIN_FETCH_MINUTES and last_10m_marker != current_hm:
                last_10m_marker = current_hm

            for symbol_info in WATCHLIST:
                
                if symbol_info["symbol"] not in state:
                    state[symbol_info["symbol"]] = default_symbol_state()
                
                try:
                    
                    process_symbol(smart_api, symbol_info, state)
                except Exception as e:
                    log.error(f"Error processing {symbol_info['symbol']}: {e}", exc_info=True)
                time.sleep(1)  # small gap between symbols to respect API rate limits

            save_state(state)
            time.sleep(LOOP_SLEEP_SECONDS)

        except KeyboardInterrupt:
            log.info("Bot stopped manually.")
            save_state(state)
            break
        except Exception as e:
            log.error(f"Main loop error: {e}", exc_info=True)
            send_telegram(f"⚠️ Bot main loop error: {e}")
            time.sleep(30)


if __name__ == "__main__":
    main()

2026-07-20 13:09:31,557 [INFO] Starting bot. DRY_RUN=False
[I 260720 13:09:31 smartConnect:124] in pool
2026-07-20 13:09:32,478 [INFO] Logged in to Angel One SmartAPI.
2026-07-20 13:09:32,489 [INFO] Loaded existing state from disk (resuming after restart).


                       time      open      high       low     close  volume
0 2026-07-20 12:30:00+05:30  77513.00  77535.50  77501.09  77512.69     0.0
1 2026-07-20 12:35:00+05:30  77515.09  77524.07  77461.82  77473.40     0.0
2 2026-07-20 12:40:00+05:30  77473.73  77493.78  77444.84  77444.84     0.0
3 2026-07-20 12:45:00+05:30  77442.83  77455.43  77368.29  77388.12     0.0
4 2026-07-20 12:50:00+05:30  77387.52  77478.06  77387.52  77478.06     0.0
5 2026-07-20 12:55:00+05:30  77478.54  77556.83  77474.53  77551.70     0.0
6 2026-07-20 13:00:00+05:30  77548.01  77559.36  77521.91  77542.61     0.0
7 2026-07-20 13:05:00+05:30  77543.18  77546.01  77509.50  77524.70     0.0
                       time      open      high       low     close  volume
0 2026-07-20 12:30:00+05:30  77513.00  77535.50  77501.09  77512.69     0.0
1 2026-07-20 12:35:00+05:30  77515.09  77524.07  77461.82  77473.40     0.0
2 2026-07-20 12:40:00+05:30  77473.73  77493.78  77444.84  77444.84     0.0
3 2026-07-20

2026-07-20 13:10:04,029 [ERROR] Candle fetch error (attempt 1): Couldn't parse the JSON response received from the server: b'Access denied because of exceeding access rate'


                       time      open      high       low     close  volume
0 2026-07-20 12:30:00+05:30  77513.00  77535.50  77501.09  77512.69     0.0
1 2026-07-20 12:35:00+05:30  77515.09  77524.07  77461.82  77473.40     0.0
2 2026-07-20 12:40:00+05:30  77473.73  77493.78  77444.84  77444.84     0.0
3 2026-07-20 12:45:00+05:30  77442.83  77455.43  77368.29  77388.12     0.0
4 2026-07-20 12:50:00+05:30  77387.52  77478.06  77387.52  77478.06     0.0
5 2026-07-20 12:55:00+05:30  77478.54  77556.83  77474.53  77551.70     0.0
6 2026-07-20 13:00:00+05:30  77548.01  77559.36  77521.91  77542.61     0.0
7 2026-07-20 13:05:00+05:30  77543.18  77546.01  77509.50  77530.49     0.0
8 2026-07-20 13:10:00+05:30  77529.49  77529.65  77525.10  77526.23     0.0
                       time      open      high       low     close  volume
0 2026-07-20 12:30:00+05:30  77513.00  77535.50  77501.09  77512.69     0.0
1 2026-07-20 12:35:00+05:30  77515.09  77524.07  77461.82  77473.40     0.0
2 2026-07-20

2026-07-20 13:10:36,638 [ERROR] Candle fetch error (attempt 1): Couldn't parse the JSON response received from the server: b'Access denied because of exceeding access rate'
2026-07-20 13:10:37,859 [ERROR] Candle fetch error (attempt 2): Couldn't parse the JSON response received from the server: b'Access denied because of exceeding access rate'


...

                       time      open      high       low     close  volume
0 2026-07-20 13:10:00+05:30  77529.49  77651.80  77514.95  77651.80     0.0
1 2026-07-20 13:15:00+05:30  77657.16  77685.72  77640.27  77668.59     0.0
2 2026-07-20 13:20:00+05:30  77669.68  77721.04  77663.55  77717.20     0.0
3 2026-07-20 13:25:00+05:30  77715.97  77754.41  77688.97  77743.57     0.0
4 2026-07-20 13:30:00+05:30  77745.28  77755.26  77717.88  77729.32     0.0
5 2026-07-20 13:35:00+05:30  77728.92  77759.91  77717.42  77722.97     0.0
6 2026-07-20 13:40:00+05:30  77722.95  77730.97  77694.34  77706.01     0.0
7 2026-07-20 13:45:00+05:30  77705.47  77758.53  77697.39  77758.53     0.0
                       time      open      high       low     close  volume
0 2026-07-20 13:10:00+05:30  77529.49  77651.80  77514.95  77651.80     0.0
1 2026-07-20 13:15:00+05:30  77657.16  77685.72  77640.27  77668.59     0.0
2 2026-07-20 13:20:00+05:30  77669.68  77721.04  77663.55  77717.20     0.0
3 2026-07-20

2026-07-20 13:47:39,632 [ERROR] Candle fetch error (attempt 1): Couldn't parse the JSON response received from the server: b'Access denied because of exceeding access rate'


                       time      open      high       low     close  volume
0 2026-07-20 13:10:00+05:30  77529.49  77651.80  77514.95  77651.80     0.0
1 2026-07-20 13:15:00+05:30  77657.16  77685.72  77640.27  77668.59     0.0
2 2026-07-20 13:20:00+05:30  77669.68  77721.04  77663.55  77717.20     0.0
3 2026-07-20 13:25:00+05:30  77715.97  77754.41  77688.97  77743.57     0.0
4 2026-07-20 13:30:00+05:30  77745.28  77755.26  77717.88  77729.32     0.0
5 2026-07-20 13:35:00+05:30  77728.92  77759.91  77717.42  77722.97     0.0
6 2026-07-20 13:40:00+05:30  77722.95  77730.97  77694.34  77706.01     0.0
7 2026-07-20 13:45:00+05:30  77705.47  77777.52  77697.39  77777.52     0.0
                       time      open      high       low     close  volume
0 2026-07-20 13:10:00+05:30  77529.49  77651.80  77514.95  77651.80     0.0
1 2026-07-20 13:15:00+05:30  77657.16  77685.72  77640.27  77668.59     0.0
2 2026-07-20 13:20:00+05:30  77669.68  77721.04  77663.55  77717.20     0.0
3 2026-07-20

2026-07-20 13:48:12,337 [ERROR] Candle fetch error (attempt 1): Couldn't parse the JSON response received from the server: b'Access denied because of exceeding access rate'
2026-07-20 13:48:13,535 [ERROR] Candle fetch error (attempt 2): Couldn't parse the JSON response received from the server: b'Access denied because of exceeding access rate'


                       time      open      high       low     close  volume
0 2026-07-20 13:10:00+05:30  77529.49  77651.80  77514.95  77651.80     0.0
1 2026-07-20 13:15:00+05:30  77657.16  77685.72  77640.27  77668.59     0.0
2 2026-07-20 13:20:00+05:30  77669.68  77721.04  77663.55  77717.20     0.0
3 2026-07-20 13:25:00+05:30  77715.97  77754.41  77688.97  77743.57     0.0
4 2026-07-20 13:30:00+05:30  77745.28  77755.26  77717.88  77729.32     0.0
5 2026-07-20 13:35:00+05:30  77728.92  77759.91  77717.42  77722.97     0.0
6 2026-07-20 13:40:00+05:30  77722.95  77730.97  77694.34  77706.01     0.0
7 2026-07-20 13:45:00+05:30  77705.47  77785.47  77697.39  77780.25     0.0
                       time      open      high       low     close  volume
0 2026-07-20 13:10:00+05:30  77529.49  77651.80  77514.95  77651.80     0.0
1 2026-07-20 13:15:00+05:30  77657.16  77685.72  77640.27  77668.59     0.0
2 2026-07-20 13:20:00+05:30  77669.68  77721.04  77663.55  77717.20     0.0
3 2026-07-20